In [58]:
%pip install xlrd

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 96 kB 9.6 MB/s  eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [74]:
from sentence_transformers import SentenceTransformer
import pandas as pd
from models.models import BertweetClassifier
from preprocessing_steps.data_cleanup import *
from sklearn.model_selection import train_test_split
import pytorch_lightning as pl
import torch
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader

In [90]:

class BertweetClassifier(pl.LightningModule):
    def __init__(
        self,
        model_name: str = "vinai/bertweet-base",
        num_labels: int = 3,
        learning_rate: float = 5e-5,
        warmup_ratio: float = 0.10,
        freeze_encoder: bool = True,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.warmup_ratio = warmup_ratio

        # 1) encoder
        self.encoder = SentenceTransformer(model_name)

        # 2) simple linear head
        hidden = self.encoder.get_sentence_embedding_dimension()
        self.classifier = nn.Linear(hidden, num_labels)

        # 3) (optionally) freeze encoder
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        self.loss_fn = nn.CrossEntropyLoss()

    # Forward expects ONLY the list[str] texts
    def forward(self, texts: list[str]):
        embeds = self.encoder.encode(texts, convert_to_tensor=True, device=self.device)
        return self.classifier(embeds)

    # ------------------------------------------------------------------
    # Shared step to avoid duplication
    def _shared_step(self, batch, stage: str):
        texts, labels = batch                    # <- from our collate_fn
        logits = self(texts)
        loss   = self.loss_fn(logits, labels.to(self.device))

        preds  = logits.argmax(1)
        acc    = (preds == labels.to(self.device)).float().mean()

        # nice logging
        self.log(f"{stage}_loss", loss,  prog_bar=True, on_step=(stage=="train"), on_epoch=True)
        self.log(f"{stage}_acc",  acc,   prog_bar=True, on_epoch=True,   on_step=False)
        # print(f"{stage}_loss", loss,  prog_bar=True, on_step=(stage=="train"), on_epoch=True)
        # print(f"{stage}_acc",  acc,   prog_bar=True, on_epoch=True,   on_step=False)
        return loss


    # Lightning API -----------------------------------------------------
    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        # Re-use the same bookkeeping you used for val:
        self._shared_step(batch, stage="test")
        
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        total_steps  = self.trainer.estimated_stepping_batches
        warmup_steps = int(self.hparams.warmup_ratio * total_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        return {
            "optimizer":  optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",      # step every batch
                "frequency": 1,
                "name": "linear-warmup",
            },
        }
    
def get_dataloader_labeled(df=None, text_column='clean_tweet', target=None, shuffle=True, batch_size=64, drop_last=True):
  data = list(zip(df[text_column].tolist(), df[target].tolist()))
  dataloader = DataLoader(df, shuffle=shuffle, batch_size=batch_size,
                          collate_fn=lambda x:x, drop_last=drop_last)
  return dataloader


class TweetsDataModule(pl.LightningDataModule):
    def __init__(self, data: pd.DataFrame, batch_size: int = None, target_col: str = 'AR'):
        super().__init__()
        self.data = data.copy()
        self.batch_size = batch_size
        self.target_col = target_col
    
    def setup(self, stage: str):
        # Assign train/val datasets for use in dataloaders
        self.data = preprocess_text(self.data, text_col="text")
        # self.data = self.data[['clean_text', self.target_col]]

        self.train, self.test = train_test_split(self.data, train_size=0.8, random_state=2025, shuffle=True)
        self.train, self.val = train_test_split(self.train, train_size=0.8, random_state=2025, shuffle=True)
    
    def train_dataloader(self):
        #  NO super().__init__() here
        data = list(zip(self.train["clean_text"], self.train[self.target_col]))
        return DataLoader(
            data,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=lambda batch: ( [t for t, _ in batch],
                                        torch.tensor([l for _, l in batch]) ),
            drop_last=True,
        )

    def val_dataloader(self):
        data = list(zip(self.val["clean_text"], self.val[self.target_col]))
        return DataLoader(
            data,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=lambda batch: ( [t for t, _ in batch],
                                        torch.tensor([l for _, l in batch]) ),
        )

    def test_dataloader(self):
        data = list(zip(self.test["clean_text"], self.test[self.target_col]))
        return DataLoader(
            data,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=lambda batch: ( [t for t, _ in batch],
                                        torch.tensor([l for _, l in batch]) ),
        )
    


tweets_labeled = pd.read_csv('data/training_data.csv')
tweets_labeled['AR'].replace({4:2}, inplace=True)
dataModule = TweetsDataModule(tweets_labeled, 32)
model_name = "digio/Twitter4SSE"
model_name2 = "peulsilva/sentence-transformer-trained-tweet"
model = BertweetClassifier(model_name=model_name2, learning_rate=5e-5)
trainer = pl.Trainer(max_epochs=5)
print(f"dataModule: {dataModule}")
trainer.fit(model=model, datamodule=dataModule)


/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_59298/1050883580.py:139: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  tweets_labeled['AR'].replace({4:2}, inplace=True)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


dataModule: {Train dataloader: None}
{Validation dataloader: None}
{Test dataloader: None}
{Predict dataloader: None}


Loading `train_dataloader` to estimate number of stepping batches.
/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

  | Name       | Type                | Params | Mode 
-----------------------------------------------------------
0 | encoder    | SentenceTransformer | 109 M  | train
1 | classifier | Linear              | 2.3 K  | train
2 | loss_fn    | CrossEntropyLoss    | 0      | train
-----------------------------------------------------------
2.3 K     Trainable params
109 M     Non-trainable params
109 M     Total params
437.938   Total estimated model params size (MB)
5         Modules in train mode
228       Modules in eval mode


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 4: 100%|██████████| 74/74 [00:34<00:00,  2.17it/s, v_num=36, train_loss_step=0.442, val_loss=0.414, val_acc=0.427, train_loss_epoch=0.410, train_acc=0.386]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 74/74 [00:34<00:00,  2.12it/s, v_num=36, train_loss_step=0.442, val_loss=0.414, val_acc=0.427, train_loss_epoch=0.410, train_acc=0.386]


In [91]:
# 3epoch 5e-5lr: [{'test_loss': 0.43339282274246216, 'test_acc': 0.35087719559669495}]
# 5epoch 5e-5lr: [{'test_loss': 0.4054107964038849, 'test_acc': 0.36572200059890747}]
# 5epoch 1e-4lr: [{'test_loss': 0.4054107964038849, 'test_acc': 0.36572200059890747}]

test_metrics = trainer.test(model=model, datamodule=dataModule)      # <- returns a list of dicts
print(test_metrics)

/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 24/24 [00:07<00:00,  3.20it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.36302295327186584    │
│         test_loss         │    0.38623976707458496    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.38623976707458496, 'test_acc': 0.36302295327186584}]


In [68]:
preds = trainer.predict(model=model, datamodule=dm)

None


Label mapping 
Problem: 0
Soution: 1
Other: 2

In [ ]:
def get_labels(df, col_name):
    conditions = [
        df[col_name] == 0,
        df[col_name] == 1,
        df[col_name] == 2
    ]
    choices = ['Problem', 'Solution', 'Other']
    return np.select(conditions, choices, default=np.NAN)

In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split
import pytorch_lightning as pl
import torch
from torch.utils.data import DataLoader

# ----------------------------------------------------------------------
# dummy preprocess -----------------------------------------------------
def preprocess_text(df, text_col="text"):
    df = df.copy()
    df["clean_text"] = df[text_col].str.lower()
    return df

# ----------------------------------------------------------------------
# small helper ---------------------------------------------------------
def get_dataloader_labeled(df, text_column="clean_text", target="AR",
                           shuffle=True, batch_size=32, drop_last=True):
    data = list(zip(df[text_column].tolist(), df[target].tolist()))
    return DataLoader(
        data,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=lambda x: x,
        drop_last=drop_last,
    )

# ----------------------------------------------------------------------
# LightningDataModule --------------------------------------------------
class TweetsDataModule(pl.LightningDataModule):
    def __init__(self, data: pd.DataFrame, batch_size: int = 32,
                 target_col: str = "AR"):
        super().__init__()
        self.data        = data.copy()
        self.batch_size  = batch_size
        self.target_col  = target_col

    # Lightning will call this before asking for the dataloaders
    def setup(self, stage=None):
        self.data = preprocess_text(self.data, text_col="text")
        self.data = self.data[["clean_text", "AR", "MB"]]

        tmp, self.test = train_test_split(self.data,   test_size=0.2, random_state=2025)
        self.train, self.val = train_test_split(tmp,   test_size=0.2, random_state=2025)

    # ---------------- required by Lightning ---------------------------
    def train_dataloader(self):     # <-- Lightning looks for THIS name
        return get_dataloader_labeled(self.train,
                                      text_column="clean_text",
                                      target=self.target_col,
                                      shuffle=True,
                                      batch_size=self.batch_size)

    def val_dataloader(self):
        return get_dataloader_labeled(self.val,
                                      text_column="clean_text",
                                      target=self.target_col,
                                      shuffle=False,
                                      batch_size=self.batch_size,
                                      drop_last=False)

    def test_dataloader(self):
        return get_dataloader_labeled(self.test,
                                      text_column="clean_text",
                                      target=self.target_col,
                                      shuffle=False,
                                      batch_size=self.batch_size,
                                      drop_last=False)

# quick self-test — should print True
# dm = TweetsDataModule(pd.DataFrame({"text": ["a", "b"], "AR": [0, 1], "MB": [1, 0]}))

dm    = TweetsDataModule(tweets_labeled, batch_size=32)
print(pl.utilities.model_helpers.is_overridden("train_dataloader", dm))
model = BertweetClassifier(model_name="peulsilva/sentence-transformer-trained-tweet")

trainer = pl.Trainer(limit_train_batches=100, max_epochs=1)

# correct: the datamodule is passed through the *named* parameter
trainer.fit(model, None, dm)


True


NameError: name 'nn' is not defined

In [ ]:
# tweets_labeled = preprocess_text(tweets_labeled, text_col="text")

# tweets_labeled = tweets_labeled[['tweet_id', 'clean_text', 'AR','MB']]

# train_set, test_set = train_test_split(tweets_labeled[['clean_text', 'AR']],
#                                              train_size=0.8, random_state=2025, shuffle=True)

# train_set, validation_set = train_test_split(train_set, train_size=0.8, random_state=2025, shuffle=True)

In [44]:
model_name = "digio/Twitter4SSE"
model_name2 = "peulsilva/sentence-transformer-trained-tweet"
model = BertweetClassifier(model_name=model_name2)
trainer = pl.Trainer(limit_train_batches=100, max_epochs=1)
trainer.fit(model, datamodule=dataModule)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


KeyError: 'text'

# Data process

In [104]:
import os
path = 'data/RandomSampledTweets/'

dfs = []
for file in os.listdir(path):
    print(file)
    source = "randomized_" + file[-13:-9]
    if 'Done' in file:
        full_path = os.path.join(path,file)
        df = pd.read_csv(full_path)
        df = df[['tweet_id', 'text', 'author_id', 'tw_date', 'year', 'AR', 'MB']]
        df['tw_date'] = pd.to_datetime(df['tw_date'])
        df['author_id'] = df['author_id'].astype(str)
        df['source'] = source
        # df['tweet_id'] = df['tweet_id'].astype(str)
        dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data.head()

random_tweets_2016_Done.csv
random_tweets_2021_Done.csv
random_tweets_2014_Done.csv
random_tweets_2015_Done.csv
random_tweets_2018_Done.csv
random_tweets_2013_Done.csv
random_tweets_2012_Done.csv


,tweet_id,text,author_id,tw_date,year,AR,MB,source
0,7.300000e+17,Congress has responsibility to protect familie...,1061029050,2016-05-11,2016,1,1,randomized_2016
1,7.300000e+17,Met w/folks from General Aviation Mfg Assoc. a...,1061029050,2016-05-11,2016,2,2,randomized_2016
2,7.080000e+17,Talked about the benefits of bicycling for ND ...,1061029050,2016-03-09,2016,3,3,randomized_2016
3,7.820000e+17,At the Dick Morris Memorial Tailgate for the U...,10615232,2016-10-01,2016,3,3,randomized_2016
4,8.040000e+17,SOON: I will speak on the #SenateFloor on the ...,1071402577,2016-11-30,2016,3,3,randomized_2016


In [123]:
data['split'] = 'train'  # Default all to train

# Group by year and sample 50 for the test set
test_indices = (
    data.groupby('year', group_keys=False)
      .apply(lambda x: x.sample(n=min(50, len(x)), random_state=42))
      .index
)
data.loc[test_indices, 'split'] = 'test'

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_20388/3707914531.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data.groupby('year', group_keys=False)


In [129]:
data_train = data[data['split']=='train']

In [127]:
test_data = data[data['split']=='test']
test_data.drop('split',axis=1, inplace=True)
test_data.head()

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_20388/3092706159.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data.drop('split',axis=1, inplace=True)


,tweet_id,text,author_id,tw_date,year,AR,MB,source
0,7.300000e+17,Congress has responsibility to protect familie...,1061029050,2016-05-11,2016,1,1,randomized_2016
5,8.070000e+17,"Last night, my CHIP IN for Vets bill was calle...",1071402577,2016-12-08,2016,2,2,randomized_2016
9,8.100000e+17,Great food/convo about north Omaha biz &amp; g...,1071402577,2016-12-16,2016,1,3,randomized_2016
15,7.550000e+17,Talking about strategies to keep #AsianCarp ou...,1074518754,2016-07-19,2016,1,1,randomized_2016
22,7.970000e+17,"Today, I met with local veterans &amp; service...",1099199839,2016-11-10,2016,3,3,randomized_2016


In [134]:
test_data.year.value_counts()

year
2016    50
2021    50
2014    50
2015    50
2018    50
2013    50
2012    50
Name: count, dtype: int64

In [128]:
test_data.to_csv("data/test_data.csv")

In [108]:
str(data['tweet_id'].tolist()[0])

'7.3e+17'

In [120]:
import numpy as np
df_path = './data/MichelleCoding1500 corrected march_13_2025.xlsx'
df_1500 = pd.read_excel(df_path)
df_1500['date'] = pd.to_datetime(df_1500['date'])
df_1500 = df_1500[['id', 'text', 'author_id', 'date', 'year', 'AR', 'MB']]
df_1500.rename(columns={'id':'tweet_id', 'date':'tw_date'}, inplace=True)
df_1500['source'] = np.where(df_1500['year'] == 2020, 'selected_2020', 'selected_2017')
df_1500.head()

,tweet_id,text,author_id,tw_date,year,AR,MB,source
0,908481742163517056,Please join in building support and momentum f...,Lindsey Graham,2017-09-15 00:05:00+00:00,2017,2,2,selected_2017
1,908482907320339968,.@FLOIR_comm has updated its ‚ÄúHurricane Seas...,Rick Scott,2017-09-15 00:09:38+00:00,2017,2,2,selected_2017
2,908483151206509952,RT @HealthyFla: #FLHealth urges caution with s...,Rick Scott,2017-09-15 00:10:36+00:00,2017,1,1,selected_2017
3,908483250032692992,Kim Jong-un's missile launches show how much h...,James Lankford,2017-09-15 00:11:00+00:00,2017,1,1,selected_2017
4,908483923109448960,@realDonaldTrump's refusal to hold white supre...,Bob Casey,2017-09-15 00:13:40+00:00,2017,1,1,selected_2017


In [139]:
df_1500['split'] = 'train'  # Default all to train

# Group by year and sample 50 for the test set
test_indices = (
    df_1500.groupby('year', group_keys=False)
      .apply(lambda x: x.sample(n=min(50, len(x)), random_state=42))
      .index
)
df_1500.loc[test_indices, 'split'] = 'test'
df_1500_train = df_1500[df_1500['split']=='train'].drop('split', axis=1)
df_1500_test = df_1500[df_1500['split']=='test'].drop('split', axis=1)

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_20388/1607151548.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_1500.groupby('year', group_keys=False)


In [109]:
df_2019 = pd.read_excel('data/2019_updated.xls')
df_2019['id'] = df_2019['id']
df_2019['created_at'] = pd.to_datetime(df_2019['created_at'])
df_2019['year'] = df_2019['created_at'].dt.year
df_2019 = df_2019[['id', 'text', 'author_id', 'created_at', 'year', 'AR', 'MB']]
df_2019.rename(columns={'id':'tweet_id', 'created_at':'tw_date'}, inplace=True)
df_2019['source'] = 'selected_2019'
df_2019.head()

,tweet_id,text,author_id,tw_date,year,AR,MB,source
0,1.080000e+18,Wishing everyone a safe #NewYearsEve and a hap...,Bob Menendez,2019-01-01 00:00:21+00:00,2019,3,3,selected_2019
1,1.080000e+18,Just a few. I am sure others have their nomine...,John Cornyn,2019-01-01 00:00:27+00:00,2019,3,3,selected_2019
2,1.080000e+18,IÇƒÙm proud of what weÇƒÙve accomplished for N...,Catherine Cortez Masto,2019-01-01 00:14:50+00:00,2019,2,2,selected_2019
3,1.080000e+18,"As your next senior senator, I enter the 116th...",Catherine Cortez Masto,2019-01-01 00:15:15+00:00,2019,2,1,selected_2019
4,1.080000e+18,"Out of a tragedy, some great news: with the Pr...",Mark Warner,2019-01-01 00:59:26+00:00,2019,2,2,selected_2019


In [143]:
df_2019['split'] = 'train'  # Default all to train

# Group by year and sample 50 for the test set
test_indices = (
    df_2019.apply(lambda x: x.sample(n=min(50, len(x)), random_state=42))
      .index
)
df_2019.loc[test_indices, 'split'] = 'test'
df_2019_train = df_2019[df_2019['split']=='train'].drop('split', axis=1)
df_2019_test = df_2019[df_2019['split']=='test'].drop('split', axis=1)

In [145]:
all_test=pd.concat([test_data, df_2019_test, df_1500_test])

In [148]:
all_test.year.value_counts().sort_index()
all_test.to_csv('data/test_data.csv')

In [149]:
df_all_labeled = pd.concat([data_train, df_2019_train, df_1500_train])

In [150]:
df_all_labeled.source.value_counts().sort_index()

source
randomized_2012    121
randomized_2013    127
randomized_2014    224
randomized_2015    206
randomized_2016    316
randomized_2018    418
randomized_2021    408
selected_2017      951
selected_2019      484
selected_2020      449
Name: count, dtype: int64

In [151]:
df_all_labeled.year.value_counts().sort_index()

year
2012    121
2013    127
2014    224
2015    206
2016    316
2017    951
2018    418
2019    484
2020    449
2021    408
Name: count, dtype: int64

In [152]:
df_all_labeled.to_csv('data/training_data.csv')

In [111]:
data.shape

(2170, 8)

In [113]:
df_1500.year.value_counts().sort_index()

year
2017    1001
2020     499
Name: count, dtype: int64